# 06 — Final Test Set Evaluation
### Heads-Up | Stage 7: Evaluation — Final (part 6 of 6)

**This is the one and only time the test set is opened.** Everything up to this point, all model comparisons, all tuning, all threshold selection, has used train and validation only. This notebook applies the final chosen model and threshold to `model_ready_test.csv` exactly once, and reports the result honestly, whatever it is.


## 0. Leakage audit confirmation, before opening test data

A final sanity check against `docs/data_dictionary.md`, confirming no post-outcome field made it into the feature set, before test data is touched.


In [ ]:
import pandas as pd
from pathlib import Path

test_df = pd.read_csv(Path('../../../data/processed/model_ready_test.csv'))

leakage_fields = ['Delivery Status', 'Days for shipping (real)', 'shipping date (DateOrders)']
present_leakage_fields = [c for c in leakage_fields if c in test_df.columns]

assert len(present_leakage_fields) == 0, f"Leakage fields found in feature set: {present_leakage_fields}"
print("Leakage audit passed — no post-outcome fields present in the feature set.")
print(f"Test set shape: {test_df.shape}")


## 1. Load the final chosen model and threshold

In [ ]:
import joblib
import json

# Update this to whichever model actually won after tuning (Notebooks 01–03) — placeholder assumes Random Forest
final_model = joblib.load('../../../models/random_forest_tuned.pkl')

with open('../../../data/processed/chosen_threshold.json') as f:
    threshold_info = json.load(f)
CHOSEN_THRESHOLD = threshold_info['threshold']
print(f"Using threshold: {CHOSEN_THRESHOLD}")

bool_cols = test_df.select_dtypes(include='bool').columns
test_df[bool_cols] = test_df[bool_cols].astype(int)

X_test = test_df.drop(columns=['Late_delivery_risk'])
y_test = test_df['Late_delivery_risk']


## 2. Evaluate on the test set, at both the default and the chosen threshold

In [ ]:
from sklearn.metrics import (recall_score, precision_score, f1_score, roc_auc_score,
                              average_precision_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

y_score_test = final_model.predict_proba(X_test)[:, 1]

for label, thresh in [('Default (0.5)', 0.5), (f'Cost-optimal ({CHOSEN_THRESHOLD:.2f})', CHOSEN_THRESHOLD)]:
    y_pred_test = (y_score_test >= thresh).astype(int)
    print(f"--- {label} ---")
    print(f"Recall: {recall_score(y_test, y_pred_test):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
    print(f"F1: {f1_score(y_test, y_pred_test):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_score_test):.4f}")
    print(f"PR-AUC: {average_precision_score(y_test, y_score_test):.4f}")
    print()


**What we found:** *(fill in — this is the true, final, generalization performance of the project. Compare test-set numbers against validation-set numbers for the same model/threshold: a close match confirms the model generalizes well and validation wasn't overly optimistic; a meaningful drop is worth reporting honestly as a limitation, not hidden.)*


## 3. Full evaluation visuals at the chosen threshold

In [ ]:
y_pred_final = (y_score_test >= CHOSEN_THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['On time', 'Late'], yticklabels=['On time', 'Late'])
axes[0].set_title('Final Model — Test Set Confusion Matrix')

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_score_test)
axes[1].plot(fpr, tpr, color='#4C72B0', linewidth=2)
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('Final Model — Test Set ROC Curve')

plt.tight_layout()
plt.savefig('../../../reports/figures/final_test_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

print(classification_report(y_test, y_pred_final, target_names=['On time (0)', 'Late (1)']))


**What we found:** *(fill in — this is the headline result for Stage 9's recommendation. State the final Recall/Precision plainly, and translate the confusion matrix into business language: how many actual late orders would this system catch, and how many false alarms would operations staff need to tolerate for that.)*


## 4. Save final results

In [ ]:
final_results = {
    'model': type(final_model).__name__,
    'threshold': CHOSEN_THRESHOLD,
    'test_recall': float(recall_score(y_test, y_pred_final)),
    'test_precision': float(precision_score(y_test, y_pred_final)),
    'test_f1': float(f1_score(y_test, y_pred_final)),
    'test_roc_auc': float(roc_auc_score(y_test, y_score_test)),
    'test_pr_auc': float(average_precision_score(y_test, y_score_test)),
}
with open('../../../data/processed/final_test_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Saved final test results.")
print("\nStage 7/8 complete. Ready for Stage 9: Recommendation.")
